## Example 1: Basic Fiona Setup and Vector Format Support

In [ ]:
import fiona
from fiona import crs
import json
from io import StringIO

# Example 1: Basic Fiona Setup and Vector Format Support
print("Example 1: Basic Fiona Setup and Vector Format Support")

try:
    # Check fiona version and supported drivers
    print(f"✓ Fiona version: {fiona.__version__}")
    
    # List supported vector formats
    supported_drivers = fiona.supported_drivers
    print(f"\n✓ Supported drivers ({len(supported_drivers)}):")
    
    # Show key vector formats
    important_formats = ['ESRI Shapefile', 'GeoJSON', 'GPKG', 'CSV', 'KML']
    for format_name in important_formats:
        if format_name in supported_drivers:
            access_modes = supported_drivers[format_name]
            print(f"   {format_name}: {access_modes}")
        else:
            print(f"   {format_name}: Not available")
    
    # Check coordinate reference system support
    print(f"\n✓ CRS (Coordinate Reference System) support:")
    
    # Common CRS examples
    crs_examples = {
        'WGS84': 'EPSG:4326',
        'Web Mercator': 'EPSG:3857',
        'UTM Zone 33N': 'EPSG:32633'
    }
    
    for crs_name, epsg_code in crs_examples.items():
        try:
            crs_info = crs.from_epsg(int(epsg_code.split(':')[1]))
            print(f"   {crs_name} ({epsg_code}): ✓")
        except Exception as e:
            print(f"   {crs_name} ({epsg_code}): Error - {e}")
            raise
    
    # Environment information
    print(f"\n✓ Fiona environment:")
    env_info = fiona.env.Env()
    with env_info:
        # Get GDAL version (underlying library)
        gdal_version = fiona._env.get_gdal_version_tuple()
        print(f"   GDAL version: {'.'.join(map(str, gdal_version))}")
        
        # Show available format drivers
        driver_count = len(fiona.drvsupport.supported_drivers)
        print(f"   Available format drivers: {driver_count}")
    
    # Show file format capabilities
    print(f"\n✓ Format capabilities:")
    format_capabilities = {
        'Shapefile (.shp)': 'Read/Write, most common GIS format',
        'GeoJSON (.geojson)': 'Read/Write, web-friendly format',
        'GeoPackage (.gpkg)': 'Read/Write, SQLite-based format',
        'KML (.kml)': 'Read/Write, Google Earth format',
        'CSV (.csv)': 'Read/Write, with geometry as WKT'
    }
    
    for format_name, description in format_capabilities.items():
        print(f"   {format_name}: {description}")
    
    # Show geometry types supported
    print(f"\n✓ Supported geometry types:")
    geometry_types = [
        'Point', 'MultiPoint', 'LineString', 'MultiLineString',
        'Polygon', 'MultiPolygon', 'GeometryCollection'
    ]
    
    for geom_type in geometry_types:
        print(f"   {geom_type}")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"fiona example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 completed")

## Example 2: Creating and Writing Geospatial Data

In [ ]:
import fiona
from fiona import crs, transform
import tempfile
import os

# Example 2: Creating and Writing Geospatial Data
print("Example 2: Creating and Writing Geospatial Data")

try:
    # Create sample geospatial data in memory
    print("✓ Creating sample geospatial features:")
    
    # Define schema for our features
    schema = {
        'geometry': 'Point',
        'properties': {
            'id': 'int',
            'name': 'str',
            'population': 'int',
            'elevation': 'float'
        }
    }
    
    # Sample city data
    cities = [
        {
            'geometry': {'type': 'Point', 'coordinates': [-74.0059, 40.7128]},
            'properties': {'id': 1, 'name': 'New York', 'population': 8400000, 'elevation': 10.0}
        },
        {
            'geometry': {'type': 'Point', 'coordinates': [-118.2437, 34.0522]},
            'properties': {'id': 2, 'name': 'Los Angeles', 'population': 3900000, 'elevation': 71.0}
        },
        {
            'geometry': {'type': 'Point', 'coordinates': [-87.6298, 41.8781]},
            'properties': {'id': 3, 'name': 'Chicago', 'population': 2700000, 'elevation': 182.0}
        }
    ]
    
    print(f"   Created {len(cities)} city features")
    
    # Write to different formats
    with tempfile.TemporaryDirectory() as temp_dir:
        
        # Write as GeoJSON
        geojson_path = os.path.join(temp_dir, 'cities.geojson')
        print(f"\n✓ Writing GeoJSON file:")
        
        with fiona.open(
            geojson_path, 'w',
            driver='GeoJSON',
            schema=schema,
            crs={'init': 'epsg:4326'}
        ) as geojson_file:
            for city in cities:
                geojson_file.write(city)
        
        print(f"   GeoJSON written: {os.path.basename(geojson_path)}")
        
        # Read back and verify
        with fiona.open(geojson_path, 'r') as geojson_file:
            features = list(geojson_file)
            print(f"   Read back {len(features)} features")
            print(f"   CRS: {geojson_file.crs}")
            print(f"   Driver: {geojson_file.driver}")
        
        # Create polygon features
        polygon_schema = {
            'geometry': 'Polygon',
            'properties': {
                'id': 'int',
                'state': 'str',
                'area_km2': 'float'
            }
        }
        
        # Sample state boundaries (simplified rectangles)
        states = [
            {
                'geometry': {
                    'type': 'Polygon',
                    'coordinates': [[
                        [-125.0, 32.0], [-114.0, 32.0],
                        [-114.0, 42.0], [-125.0, 42.0],
                        [-125.0, 32.0]
                    ]]
                },
                'properties': {'id': 1, 'state': 'California', 'area_km2': 423970.0}
            }
        ]
        
        # Write polygon data
        polygon_path = os.path.join(temp_dir, 'states.geojson')
        print(f"\n✓ Writing polygon features:")
        
        with fiona.open(
            polygon_path, 'w',
            driver='GeoJSON',
            schema=polygon_schema,
            crs={'init': 'epsg:4326'}
        ) as polygon_file:
            for state in states:
                polygon_file.write(state)
        
        print(f"   Polygon file written: {os.path.basename(polygon_path)}")
        
        # Summary of created files
        print(f"\n✓ Summary of created files:")
        for file in os.listdir(temp_dir):
            file_path = os.path.join(temp_dir, file)
            file_size = os.path.getsize(file_path)
            print(f"   {file}: {file_size} bytes")
        
except Exception as e:
    print(f"Error in writing geospatial data: {e}")
    raise AssertionError(
        f"fiona example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 completed")

## Example 3: Reading and Processing Geospatial Data

In [ ]:
import fiona
import tempfile
import os
import json

# Example 3: Reading and Processing Geospatial Data
print("Example 3: Reading and Processing Geospatial Data")

try:
    # Create sample data for reading examples
    sample_data = [
        {
            'geometry': {'type': 'Point', 'coordinates': [2.3522, 48.8566]},
            'properties': {'name': 'Paris', 'country': 'France', 'population': 2161000}
        },
        {
            'geometry': {'type': 'Point', 'coordinates': [-0.1276, 51.5074]},
            'properties': {'name': 'London', 'country': 'UK', 'population': 8982000}
        },
        {
            'geometry': {'type': 'Point', 'coordinates': [13.4050, 52.5200]},
            'properties': {'name': 'Berlin', 'country': 'Germany', 'population': 3669000}
        }
    ]
    
    with tempfile.TemporaryDirectory() as temp_dir:
        
        # Create test file
        cities_path = os.path.join(temp_dir, 'european_cities.geojson')
        
        # Write cities data
        city_schema = {
            'geometry': 'Point',
            'properties': {'name': 'str', 'country': 'str', 'population': 'int'}
        }
        
        with fiona.open(
            cities_path, 'w',
            driver='GeoJSON',
            schema=city_schema,
            crs={'init': 'epsg:4326'}
        ) as cities_file:
            for city in sample_data:
                cities_file.write(city)
        
        print("✓ Sample data file created")
        
        # Reading and analyzing data
        print("\n✓ Reading city data:")
        
        with fiona.open(cities_path, 'r') as cities_file:
            print(f"   File info:")
            print(f"     Driver: {cities_file.driver}")
            print(f"     CRS: {cities_file.crs}")
            print(f"     Feature count: {len(cities_file)}")
            
            # Process each feature
            print(f"\n   City details:")
            cities_list = []
            
            for feature in cities_file:
                geom = feature['geometry']
                props = feature['properties']
                
                lon, lat = geom['coordinates']
                name = props['name']
                country = props['country']
                population = props['population']
                
                cities_list.append({
                    'name': name,
                    'country': country,
                    'population': population,
                    'coordinates': (lon, lat)
                })
                
                print(f"     {name}, {country}: {population:,} people at ({lon:.4f}, {lat:.4f})")
        
        # Data analysis
        print(f"\n✓ Data analysis:")
        
        total_population = sum(city['population'] for city in cities_list)
        avg_population = total_population / len(cities_list)
        
        print(f"   Population statistics:")
        print(f"     Total: {total_population:,}")
        print(f"     Average: {avg_population:,.0f}")
        
        # Geographic analysis
        latitudes = [city['coordinates'][1] for city in cities_list]
        longitudes = [city['coordinates'][0] for city in cities_list]
        
        print(f"\n   Geographic spread:")
        print(f"     Latitude range: {min(latitudes):.2f}° to {max(latitudes):.2f}°")
        print(f"     Longitude range: {min(longitudes):.2f}° to {max(longitudes):.2f}°")
        
except Exception as e:
    print(f"Error in reading geospatial data: {e}")
    raise AssertionError(
        f"fiona example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 completed")

## Example 4: Coordinate Reference Systems and Transformations

In [ ]:
import fiona
from fiona import crs, transform
import tempfile
import os
from math import sqrt

# Example 4: Coordinate Reference Systems and Transformations
print("Example 4: Coordinate Reference Systems and Transformations")

try:
    # CRS (Coordinate Reference System) examples
    print("✓ Working with Coordinate Reference Systems:")
    
    # Common CRS definitions
    crs_examples = {
        'WGS84': {'init': 'epsg:4326'},
        'Web Mercator': {'init': 'epsg:3857'},
        'UTM Zone 33N': {'init': 'epsg:32633'}
    }
    
    print(f"   Available CRS examples:")
    for name, crs_def in crs_examples.items():
        try:
            epsg_code = int(crs_def['init'].split(':')[1])
            crs_obj = crs.from_epsg(epsg_code)
            print(f"     {name}: {crs_def['init']} ✓")
        except Exception as e:
            print(f"     {name}: {crs_def['init']} - Error: {e}")
            raise
    
    # Create sample data in different CRS
    sample_locations = [
        {'name': 'Greenwich Observatory', 'lon': 0.0, 'lat': 51.4769},
        {'name': 'Eiffel Tower', 'lon': 2.2945, 'lat': 48.8584},
        {'name': 'Brandenburg Gate', 'lon': 13.3777, 'lat': 52.5163}
    ]
    
    with tempfile.TemporaryDirectory() as temp_dir:
        
        # Write data in WGS84 (Geographic coordinates)
        wgs84_path = os.path.join(temp_dir, 'landmarks_wgs84.geojson')
        
        schema = {
            'geometry': 'Point',
            'properties': {'name': 'str', 'original_lon': 'float', 'original_lat': 'float'}
        }
        
        # Create WGS84 data
        with fiona.open(
            wgs84_path, 'w',
            driver='GeoJSON',
            schema=schema,
            crs={'init': 'epsg:4326'}
        ) as wgs84_file:
            for location in sample_locations:
                feature = {
                    'geometry': {
                        'type': 'Point',
                        'coordinates': [location['lon'], location['lat']]
                    },
                    'properties': {
                        'name': location['name'],
                        'original_lon': location['lon'],
                        'original_lat': location['lat']
                    }
                }
                wgs84_file.write(feature)
        
        print(f"\n   WGS84 data written: {len(sample_locations)} landmarks")
        
        # Read and display original coordinates
        print(f"\n✓ Original WGS84 coordinates:")
        with fiona.open(wgs84_path, 'r') as wgs84_file:
            print(f"   CRS: {wgs84_file.crs}")
            for feature in wgs84_file:
                name = feature['properties']['name']
                lon, lat = feature['geometry']['coordinates']
                print(f"     {name}: {lon:.4f}°E, {lat:.4f}°N")
        
        # Coordinate bounds analysis
        print(f"\n✓ Coordinate bounds analysis:")
        
        with fiona.open(wgs84_path, 'r') as wgs84_file:
            lons = []
            lats = []
            
            for feature in wgs84_file:
                lon, lat = feature['geometry']['coordinates']
                lons.append(lon)
                lats.append(lat)
            
            wgs84_bounds = {
                'min_lon': min(lons),
                'max_lon': max(lons),
                'min_lat': min(lats), 
                'max_lat': max(lats)
            }
            
            print(f"   WGS84 bounds:")
            print(f"     Longitude: {wgs84_bounds['min_lon']:.4f} to {wgs84_bounds['max_lon']:.4f}")
            print(f"     Latitude: {wgs84_bounds['min_lat']:.4f} to {wgs84_bounds['max_lat']:.4f}")
        
except Exception as e:
    print(f"Error in CRS and transformations: {e}")
    raise AssertionError(
        f"fiona example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 completed")

## Example 5: Advanced Fiona Features and Performance

In [ ]:
import fiona
import tempfile
import os
import json
import time
import random
from collections import defaultdict

# Example 5: Advanced Fiona Features and Performance
print("Example 5: Advanced Fiona Features and Performance")

try:
    # Advanced geospatial data processing
    print("✓ Advanced geospatial data processing:")
    
    # Create complex multi-geometry dataset
    with tempfile.TemporaryDirectory() as temp_dir:
        
        # Create points data (infrastructure)
        points_data = [
            {'name': 'Airport_JFK', 'type': 'airport', 'coordinates': [-73.7781, 40.6413]},
            {'name': 'Port_NY', 'type': 'port', 'coordinates': [-74.0445, 40.6892]},
            {'name': 'Station_Penn', 'type': 'train', 'coordinates': [-73.9935, 40.7505]}
        ]
        
        points_path = os.path.join(temp_dir, 'infrastructure.geojson')
        point_schema = {
            'geometry': 'Point',
            'properties': {'name': 'str', 'type': 'str'}
        }
        
        with fiona.open(
            points_path, 'w',
            driver='GeoJSON',
            schema=point_schema,
            crs={'init': 'epsg:4326'}
        ) as points_file:
            for point in points_data:
                feature = {
                    'geometry': {'type': 'Point', 'coordinates': point['coordinates']},
                    'properties': {'name': point['name'], 'type': point['type']}
                }
                points_file.write(feature)
        
        print(f"   Points file created: {len(points_data)} features")
        
        # Create routes data (lines)
        routes_data = [
            {
                'name': 'Highway_I95',
                'coordinates': [[-73.7781, 40.6413], [-73.9935, 40.7505], [-74.0445, 40.6892]]
            }
        ]
        
        routes_path = os.path.join(temp_dir, 'routes.geojson')
        route_schema = {
            'geometry': 'LineString',
            'properties': {'name': 'str'}
        }
        
        with fiona.open(
            routes_path, 'w',
            driver='GeoJSON',
            schema=route_schema,
            crs={'init': 'epsg:4326'}
        ) as routes_file:
            for route in routes_data:
                feature = {
                    'geometry': {'type': 'LineString', 'coordinates': route['coordinates']},
                    'properties': {'name': route['name']}
                }
                routes_file.write(feature)
        
        print(f"   Routes file created: {len(routes_data)} features")
        
        # Analyze infrastructure by type
        print(f"\n✓ Infrastructure analysis:")
        
        with fiona.open(points_path, 'r') as points_file:
            infrastructure_types = defaultdict(list)
            
            for feature in points_file:
                infra_type = feature['properties']['type']
                infra_name = feature['properties']['name']
                infrastructure_types[infra_type].append(infra_name)
            
            print(f"   Infrastructure by type:")
            for infra_type, names in infrastructure_types.items():
                print(f"     {infra_type}: {len(names)} locations")
                for name in names:
                    print(f"       - {name}")
        
        # Performance testing with larger dataset
        print(f"\n✓ Performance testing:")
        
        # Generate random points for performance test
        num_features = 500
        large_data_path = os.path.join(temp_dir, 'performance_test.geojson')
        
        start_time = time.time()
        
        with fiona.open(
            large_data_path, 'w',
            driver='GeoJSON',
            schema={
                'geometry': 'Point',
                'properties': {'id': 'int', 'category': 'str', 'value': 'float'}
            },
            crs={'init': 'epsg:4326'}
        ) as large_file:
            
            categories = ['Type_A', 'Type_B', 'Type_C']
            
            for i in range(num_features):
                feature = {
                    'geometry': {
                        'type': 'Point',
                        'coordinates': [
                            random.uniform(-180, 180),  # Random longitude
                            random.uniform(-90, 90)     # Random latitude
                        ]
                    },
                    'properties': {
                        'id': i + 1,
                        'category': random.choice(categories),
                        'value': random.uniform(0, 100)
                    }
                }
                large_file.write(feature)
        
        write_time = time.time() - start_time
        file_size = os.path.getsize(large_data_path)
        
        print(f"   Write performance:")
        print(f"     Features written: {num_features:,}")
        print(f"     Time taken: {write_time:.3f} seconds")
        print(f"     Features per second: {num_features / write_time:.0f}")
        print(f"     File size: {file_size:,} bytes")
        
        # Read performance test
        start_time = time.time()
        
        with fiona.open(large_data_path, 'r') as large_file:
            feature_count = 0
            category_counts = defaultdict(int)
            
            for feature in large_file:
                feature_count += 1
                category = feature['properties']['category']
                category_counts[category] += 1
        
        read_time = time.time() - start_time
        
        print(f"\n   Read performance:")
        print(f"     Features read: {feature_count:,}")
        print(f"     Time taken: {read_time:.3f} seconds")
        print(f"     Features per second: {feature_count / read_time:.0f}")
        
        print(f"\n   Category distribution:")
        for category, count in sorted(category_counts.items()):
            percentage = (count / feature_count) * 100
            print(f"     {category}: {count:,} ({percentage:.1f}%)")
        
        # Summary of capabilities
        print(f"\n✓ Fiona capabilities demonstrated:")
        capabilities = [
            "Multi-geometry type handling",
            "File format support (GeoJSON)",
            "Schema definition and validation",
            "Large dataset processing",
            "Performance optimization",
            "Memory-efficient reading"
        ]
        
        for i, capability in enumerate(capabilities, 1):
            print(f"     {i}. {capability}")
        
except Exception as e:
    print(f"Error in advanced Fiona features: {e}")
    raise AssertionError(
        f"fiona example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 completed")